In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuration
SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"

# Read silver layer tables
fact_transactions = spark.table(f"{SILVER_SCHEMA}.fact_transactions")
dim_customers = spark.table(f"{SILVER_SCHEMA}.dim_customers")
dim_products = spark.table(f"{SILVER_SCHEMA}.dim_products")
dim_stores = spark.table(f"{SILVER_SCHEMA}.dim_stores")

print("✓ Silver layer tables loaded successfully")

In [0]:
ft = fact_transactions.alias("ft")
dp = dim_products.alias("dp")

gold_product_performance = (
    ft.join(dp, "product_id")
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        F.col("dp.unit_cost").alias("unit_cost"),
        F.col("dp.unit_price").alias("catalog_unit_price"),   # current price from dim_products
    )
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.sum("discount_amount").alias("total_discount_given"),
        F.count("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("store_id").alias("stores_selling"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.min("transaction_date").alias("first_sale_date"),
        F.max("transaction_date").alias("last_sale_date"),
        F.round(F.avg("ft.unit_price"), 2).alias("avg_transaction_unit_price"),  # actual price paid
    )
    .withColumn("total_cost", F.col("unit_cost") * F.col("total_units_sold"))
    .withColumn("gross_profit", F.col("total_revenue") - F.col("total_cost"))
    .withColumn("profit_margin_pct", F.round(F.col("gross_profit") / F.col("total_revenue") * 100, 2))
    .withColumn(
        "avg_discount_pct",
        F.round(F.col("total_discount_given") / (F.col("total_revenue") + F.col("total_discount_given")) * 100, 2)
    )
    .withColumn("created_at", F.current_timestamp())
)

In [0]:
# Aggregate performance by category and subcategory
gold_category_performance = (
    fact_transactions
    .join(dim_products, "product_id")
    .groupBy("category", "subcategory")
    .agg(
        F.countDistinct("product_id").alias("product_count"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.count("transaction_id").alias("transaction_count")
    )
    .withColumn(
        "revenue_per_product",
        F.round(F.col("total_revenue") / F.col("product_count"), 2)
    )
    .withColumn("created_at", F.current_timestamp())
)

# Add category revenue percentage
total_revenue = gold_category_performance.agg(F.sum("total_revenue")).collect()[0][0]

gold_category_performance = gold_category_performance.withColumn(
    "revenue_share_pct",
    F.round(F.col("total_revenue") / F.lit(total_revenue) * 100, 2)
)

print(f"Category performance records: {gold_category_performance.count():,}")
display(gold_category_performance.orderBy(F.desc("total_revenue")).limit(10))

In [0]:
# Market basket analysis - products frequently bought together
# Get products per transaction
products_per_transaction = (
    fact_transactions
    .select("transaction_id", "product_id")
    .join(dim_products.select("product_id", "product_name"), "product_id")
)

# Self-join to find product pairs
gold_product_affinity = (
    products_per_transaction.alias("p1")
    .join(
        products_per_transaction.alias("p2"),
        (F.col("p1.transaction_id") == F.col("p2.transaction_id")) &
        (F.col("p1.product_id") < F.col("p2.product_id"))  # Avoid duplicates
    )
    .groupBy(
        F.col("p1.product_id").alias("product_a_id"),
        F.col("p1.product_name").alias("product_a_name"),
        F.col("p2.product_id").alias("product_b_id"),
        F.col("p2.product_name").alias("product_b_name")
    )
    .agg(
        F.count("p1.transaction_id").alias("co_occurrence_count")
    )
    .filter(F.col("co_occurrence_count") >= 5)  # Minimum threshold
    .withColumn("created_at", F.current_timestamp())
)

print(f"Product affinity records: {gold_product_affinity.count():,}")
display(gold_product_affinity.orderBy(F.desc("co_occurrence_count")).limit(10))

In [0]:
# Compare brand performance
ft = fact_transactions.alias("ft")
dp = dim_products.alias("dp")

gold_brand_performance = (
    ft.join(dp, "product_id")
    .groupBy("brand", "category")
    .agg(
        F.countDistinct("product_id").alias("product_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.avg("ft.unit_price"), 2).alias("avg_transaction_unit_price"),  # what customers actually paid
        F.round(F.avg("dp.unit_price"), 2).alias("avg_catalog_unit_price"),      # current list price
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .withColumn(
        "revenue_per_customer",
        F.round(F.col("total_revenue") / F.col("unique_customers"), 2)
    )
    .withColumn("created_at", F.current_timestamp())
)

window_spec = Window.partitionBy("category").orderBy(F.desc("total_revenue"))
gold_brand_performance = gold_brand_performance.withColumn(
    "category_rank",
    F.row_number().over(window_spec)
)

In [0]:
# Product velocity and inventory recommendations
gold_inventory_insights = (
    fact_transactions
    .join(dim_products, "product_id")
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "subcategory"
    )
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.countDistinct("transaction_date").alias("days_with_sales"),
        F.min("transaction_date").alias("first_sale_date"),
        F.max("transaction_date").alias("last_sale_date")
    )
    .withColumn(
        "days_on_sale",
        F.datediff(F.col("last_sale_date"), F.col("first_sale_date")) + 1
    )
    .withColumn(
        "avg_daily_sales",
        F.round(F.col("total_units_sold") / F.col("days_on_sale"), 2)
    )
    .withColumn(
        "sell_through_rate",
        F.round(F.col("days_with_sales") / F.col("days_on_sale") * 100, 2)
    )
    .withColumn(
        "velocity_category",
        F.when(F.col("avg_daily_sales") >= 10, "Fast-moving")
        .when(F.col("avg_daily_sales") >= 3, "Medium-moving")
        .otherwise("Slow-moving")
    )
    .withColumn(
        "recommended_stock_days",
        F.when(F.col("velocity_category") == "Fast-moving", 30)
        .when(F.col("velocity_category") == "Medium-moving", 45)
        .otherwise(60)
    )
    .withColumn("created_at", F.current_timestamp())
)

print(f"Inventory insight records: {gold_inventory_insights.count():,}")
print("\nVelocity distribution:")
display(gold_inventory_insights.groupBy("velocity_category").count().orderBy("velocity_category"))

In [0]:
# Write all product analytics tables
gold_product_performance.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.product_performance")
print("✓ Wrote product_performance")

gold_category_performance.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.category_performance")
print("✓ Wrote category_performance")

gold_product_affinity.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.product_affinity")
print("✓ Wrote product_affinity")

gold_brand_performance.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.brand_performance")
print("✓ Wrote brand_performance")

gold_inventory_insights.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.inventory_insights")
print("✓ Wrote inventory_insights")

print("\n" + "="*50)
print("Product Analytics Gold Layer - COMPLETED")
print("="*50)